# JASPAR PWM 2025 Mouse Processing Notebook

The mouse TF matrices used to conduct this processing were accessed via the pyJASPAR package on 9/16/25. A fasta file containing 1000 base pairs upstream and 200 base pairs downstream of the TSS from each gene was constructed using the Mus_musculus.GRCm39.dna.primary_assembly.fa.gz and Mus_musculus.GRCm39.115.chr.gtf files from [Ensembl](https://useast.ensembl.org/Mus_musculus/Info/Index) and awk/bedtools. 

The motifs were then scanned using [MOODS](https://www.cs.helsinki.fi/group/pssmfind/) and associations between TFs and predicted targets were kept if there were 3+ hits or if the total score of hits exceeded 15, capping large gene sets at a size of 2500.

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np
import re
import scipy.spatial.distance as dist
import seaborn as sns
import sys
from tqdm import tqdm
import scanpy as sc

from sklearn.feature_extraction.text import TfidfVectorizer
import anndata
from collections import OrderedDict

# PWM?Sequences
from Bio import SeqIO
from MOODS import parsers
from MOODS import tools
from MOODS import scan
import pyjaspar

# Bokeh
from bokeh.io import output_notebook
from bokeh.plotting import figure, show, save, output_file
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.palettes import Category20
output_notebook()

from IPython.display import display, HTML, Markdown
sys.setrecursionlimit(100000)

## Load and Pre-Process Data

In [ ]:
def pwm_to_tuples(motif):
    return ((motif.pwm['A']), (motif.pwm['C']), (motif.pwm['G']), (motif.pwm['T']))

In [ ]:
jaspar_db = pyjaspar.jaspardb(release='JASPAR2024')
mouse_motifs = {motif.matrix_id:pwm_to_tuples(motif) for motif in jaspar_db.fetch_motifs(collection = 'CORE', species = 10090, all_versions = False)}
mouse_motif_ids = {motif.matrix_id:motif.name for motif in jaspar_db.fetch_motifs(collection = 'CORE', species = 10090, all_versions = False)}
print(f'{len(mouse_motifs)} mouse motifs')

In [ ]:
gene_info = pd.read_csv('../../tables/All_Mammalia.gene_info.gz', sep='\t', compression='gzip')
mouse_gene_info = gene_info[(gene_info['#tax_id']==10090) & (gene_info['type_of_gene']=='protein-coding')]
gene_info = gene_info[(gene_info['#tax_id']==9606) & (gene_info['type_of_gene']=='protein-coding')]
gene_info['Symbol'] = gene_info['Symbol'].str.upper()
symbols = set(gene_info['Symbol'])
geneids = gene_info.set_index('Symbol')['GeneID'].to_dict()
synonyms = gene_info.copy()
synonyms['Synonyms'] = synonyms.apply(lambda x: x['Synonyms'].upper().split('|') + [x['Symbol']], axis=1)
synonyms = synonyms.explode('Synonyms')
synonyms['Synonyms'] = synonyms['Synonyms'].str.upper()
synonyms = synonyms[synonyms['Synonyms']!='-']
synonyms = synonyms.set_index('Synonyms')['Symbol'].to_dict()

### Mouse TFs

In [ ]:
motif_matrices = []
motif_names = []

for motif_id, pwm in mouse_motifs.items():
    motif_matrices.append(pwm)
    motif_names.append(motif_id)

print(f"Prepared {len(motif_matrices)} motifs for scanning")

In [ ]:
bg = [0.25, 0.25, 0.25, 0.25]
thresholds = [tools.threshold_from_p(motif_matrix, bg, 1e-3) for motif_matrix in motif_matrices]

In [ ]:
scanner = scan.Scanner(7)
scanner.set_motifs(motif_matrices, bg, thresholds)
for i,_ in enumerate(SeqIO.parse("mouse_search_regions_1k_2h.fa", "fasta")):
    pass

from collections import defaultdict

tf_gene_hits = defaultdict(lambda: {'total_score': 0, 'n_hits': 0})

for record in tqdm(SeqIO.parse('mouse_search_regions_1k_2h.fa', 'fasta'), total=i):
    seq = str(record.seq).upper()
    seq_results = scanner.scan(seq)
    
    gene_full = record.id.split('|')[0]
    gene = gene_full.split('::')[0]
    
    for motif_idx, hits in enumerate(seq_results):
        motif = motif_names[motif_idx]
        for hit in hits:
            tf_gene_hits[(motif, gene)]['total_score'] += hit.score
            tf_gene_hits[(motif, gene)]['n_hits'] += 1

print(f'Aggregated hits for {len(tf_gene_hits)} motif-gene pairs')

In [ ]:
mouse_jaspar = pd.DataFrame([
    {'motif': motif, 'gene': gene, 'total_score': v['total_score'], 'n_hits': v['n_hits']}
    for (motif, gene), v in tf_gene_hits.items()
])
mouse_jaspar

Aim for gene set size 250≥x≤2500

In [ ]:
mouse_ensembl_genes = pd.read_csv('ensembl_mouse_gene_mapping.txt', sep='\t')
mouse_ensembl_gene_map = mouse_ensembl_genes.set_index('Gene stable ID')['Human gene name'].to_dict()
mouse_ensembl_genes

In [ ]:
mouse_associations = mouse_jaspar.copy()

mouse_associations['motif'] = mouse_associations['motif'].map(mouse_motif_ids).str.capitalize()
mouse_associations['gene'] = mouse_associations['gene'].map(mouse_ensembl_gene_map).map(synonyms).str.upper()
mouse_associations = mouse_associations.dropna()

mouse_associations = mouse_associations.groupby(['motif', 'gene']).agg(
    {'total_score':'sum', 
    'n_hits':'sum'}
).reset_index()

mouse_associations

In [ ]:
mouse_associations.to_csv('jasparpwm2025mouse_scanresults_2k1h.tsv', sep='\t')

In [ ]:
mouse_associations = pd.read_csv('jasparpwm2025mouse_scanresults_2k1h.tsv', sep='\t', index_col=0)
mouse_associations['motif'] = mouse_associations['motif'].map(lambda x: '::'.join(list(map(str.capitalize, x.split('::')))))
mouse_associations

In [ ]:
mouse_associations.nunique()

In [ ]:
counts = mouse_associations[(mouse_associations['total_score']>=15) | (mouse_associations['n_hits']>=3)].groupby(['motif'])['gene'].count()

fig, ax = plt.subplots()
sns.histplot(counts, bins=600, binwidth=25)
plt.axvline(50, 0, 300, color='black', linestyle='--')
plt.axvline(2500, 0, 300, color='black', linestyle='--')
ax.set_xlim(-250, 4000)

print(len(counts[(counts>=50) & (counts<=2500)]))
print(f'Mean: {round(counts.mean(), 3)}')
print('nunique:')
mouse_associations[(mouse_associations['total_score']>=5) & (mouse_associations['n_hits']>=5)].nunique()#.groupby(['motif'])['gene'].agg(set)

In [ ]:
mouse_trimmed = mouse_associations[(mouse_associations['total_score']>=15) | (mouse_associations['n_hits']>=3)].set_index('gene').groupby('motif')['total_score'].nlargest(2500).reset_index()
sns.histplot(mouse_trimmed.groupby('motif')['gene'].count(), bins=50)
mouse_trimmed

## Process Data for SQL

In [ ]:
human_gene_ids = gene_info.set_index('Symbol')['GeneID'].to_dict()
mouse_gene_ids = mouse_gene_info.set_index('Symbol')['GeneID'].to_dict()

In [ ]:
jaspar_mouse = mouse_trimmed.copy()
jaspar_mouse['gene'] = jaspar_mouse['gene']
jaspar_mouse = jaspar_mouse.dropna().reset_index(drop=True)
jaspar_mouse

### Dataset

In [ ]:
#(id, name, name_without_resource, description, association, gene_set_description, gene_sets_description, attribute_set_description, positive_association, negative_association, is_signed, is_continuous_valued, last_updated, directory, num_page_views, resource_fk, measurement_fk, dataset_group_fk, attribute_type_fk, attribute_group_fk, evidence_type, evidence_group, measurement_bias, attribute_type_plural)
(190, 'JASPAR Predicted Mouse Transcription Factor Targets 2025', 'Predicted Mouse Transcription Factor Targets 2025', 'Target genes of mouse transcription factors predicted using known transcription factor binding site motifs', 'target gene-transcription factor associations predicted using known transcription factor binding site motifs', 'target genes of the {0} transcription factor predicted using known transcription factor binding site motifs from the JASPAR Predicted Mouse Transcription Factor Targets 2925 dataset.', 'sets of target genes of transcription factors predicted using known transcription factor binding site motifs from the JASPAR Predicted Mouse Transcription Factor Targets 2025 dataset.', 'transcription factors regulating expression of {0} gene predicted using known transcription factor binding site motifs from the JASPAR Predicted Mouse Transcription Factor Targets 2025 dataset.', 0, 1, '2025-09-23', 'jasparpwmmouse25', 16324, 44, 1, 2, 36, 5, 'association by computational motif search', 'computational model', 'high throughput, data driven', 'transcription factors')

### Publication

In [ ]:
(278, 190, 172)

### Attributes

In [ ]:
index = 558918
attributes = pd.read_csv('../../tables/attribute.txt', sep='\t')
attributes['name_from_naming_authority'] = attributes['name_from_naming_authority'].astype(str).map(str.lower)
attribute_fks = attributes.set_index('name_from_naming_authority')['id'].to_dict()
geneinfodf = gene_info.set_index('Symbol')
motif_id_mapping = {'::'.join(list(map(str.capitalize, tf.split('::')))):motif for (motif, tf) in mouse_motif_ids.items()}

for tf in jaspar_mouse['motif'].unique():
    if tf.lower() not in attribute_fks:
        tf_id = mouse_gene_ids.get(tf.split('_')[0]) or motif_id_mapping.get(tf.split('_')[0])
        tf_url = f'https://ncbi.nlm.nih.gov/gene/{tf_id}' if type(tf_id)==int else f'https://jaspar.elixir.no/matrix/{tf_id}'
        naming_authority = 18 if type(tf_id)==int else 67
        print((index, tf, tf_id, tf_url, naming_authority), end=',\n')
        attribute_fks[tf.lower()] = index
        index += 1

### Gene Set

In [ ]:
index = 139100000
gene_set_fks = {}

for tf in jaspar_mouse['motif'].unique():
        tf_id = mouse_gene_ids.get(tf.split('_')[0]) or motif_id_mapping.get(tf.split('_')[0])
        tf_url = f'https://ncbi.nlm.nih.gov/gene/{tf_id}' if type(tf_id)==int else f'https://jaspar.elixir.no/matrix/{tf_id}'
        print((index, tf, tf_id, tf_url, 190, 36, attribute_fks[tf.lower()]), end=',\n')
        gene_set_fks[tf] = index
        index += 1

### Associations

In [ ]:
index = 95000000

genes = pd.read_csv('../../tables/gene.txt',sep='\t')
genefks = genes.set_index('symbol')['id'].to_dict()

associations = jaspar_mouse.copy()
associations['GeneSymbol'] = associations['gene'].apply(lambda x: genefks[x])
associations['TF'] = associations['motif'].apply(lambda x: gene_set_fks[x])
associations['Threshold'] = 1
associations = associations.get(['GeneSymbol', 'TF', 'total_score', 'Threshold'])
associations.columns = ['gene_fk', 'gene_set_fk', 'standardized_value', 'threshold_value']
associations = associations.reset_index(drop=True)
associations.index += index
associations = associations.rename_axis('id')
associations.to_csv('../../harmonizome-update/jasparpwmmouse25.csv')
associations

In [ ]:
associations.nunique()

## Create Downloads

In [ ]:
output_path = 'mouse_downloads'

### Binary Matrix

In [ ]:
binarymatrix = pd.crosstab(jaspar_mouse['gene'], jaspar_mouse['motif']).replace(np.nan, 0)
binarymatrixT = binarymatrix.T
binarymatrix.to_csv(f'{output_path}/gene_attribute_matrix.txt.gz', sep='\t', compression='gzip')
binarymatrix

### Gene Attribute Edge List

In [ ]:
geneids = gene_info.set_index('Symbol')['GeneID'].to_dict()
edgelist = jaspar_mouse.copy()
edgelist['Gene ID'] = edgelist['gene'].apply(lambda x: geneids[x])
edgelist['TF ID'] = edgelist['motif'].apply(lambda x: human_gene_ids.get(synonyms.get(x.split('_')[0], x)) or mouse_gene_ids.get(x.split('_')[0]) or motif_id_mapping.get(x.split('_')[0]))
edgelist['Threshold'] = 1
edgelist = edgelist[['gene', 'Gene ID', 'motif', 'TF ID', 'total_score', 'Threshold']]
edgelist.columns = ['Gene', 'Gene ID', 'TF', 'TF ID',  'Score', 'Threshold Value']
edgelist.to_csv(f'{output_path}/gene_attribute_edges.txt.gz', sep='\t', compression='gzip')
edgelist

### Gene List

In [ ]:
geneslist = edgelist[['Gene', 'Gene ID']].drop_duplicates().reset_index(drop=True)
geneslist.to_csv(f'{output_path}/gene_list_terms.txt.gz', sep='\t', compression='gzip')
geneslist

### Attribute List

In [ ]:
attributeslist = edgelist[['TF', 'TF ID']].drop_duplicates().reset_index(drop=True)
attributeslist.to_csv(f'{output_path}/attribute_list_entries.txt.gz', sep='\t', compression='gzip')
attributeslist

### Gene Set Library

In [ ]:
with open(f'{output_path}/gene_set_library_crisp.gmt', 'w') as f:
    arr = binarymatrix.reset_index(drop=True).to_numpy(dtype=np.int_)
    attributes = binarymatrix.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*binarymatrix.index[arr[:, i] == 1]])>= 5:
            print(attributes[i], '', *binarymatrix.index[arr[:, i] == 1], sep='\t', end='\n', file=f)

### Attribute Set Library

In [ ]:
with open(f'{output_path}/attribute_set_library_crisp.gmt', 'w') as f:
    arr = binarymatrixT.reset_index(drop=True).to_numpy(dtype=np.int_)
    genes = binarymatrixT.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*binarymatrixT.index[arr[:, i] == 1]])>= 5:
            print(genes[i], '', *binarymatrixT.index[arr[:, i] == 1], sep='\t', end='\n', file=f)

### Gene Similarity Matrix

In [ ]:
gene_similarity_matrix = dist.pdist(binarymatrix.to_numpy(dtype=np.int_), 'cosine')
gene_similarity_matrix = dist.squareform(gene_similarity_matrix)
gene_similarity_matrix = 1 - gene_similarity_matrix

gene_similarity_matrix = pd.DataFrame(data=gene_similarity_matrix, index=binarymatrix.index, columns=binarymatrix.index)
gene_similarity_matrix.index.name = None
gene_similarity_matrix.columns.name = None
gene_similarity_matrix.to_csv(f'{output_path}/gene_similarity_matrix_cosine.txt.gz', sep='\t', compression='gzip')
gene_similarity_matrix

### Attribute Similarity Matrix

In [ ]:
attribute_similarity_matrix = dist.pdist(binarymatrixT.to_numpy(dtype=np.int_), 'cosine')
attribute_similarity_matrix = dist.squareform(attribute_similarity_matrix)
attribute_similarity_matrix = 1 - attribute_similarity_matrix

attribute_similarity_matrix = pd.DataFrame(data=attribute_similarity_matrix, index=binarymatrixT.index, columns=binarymatrixT.index)
attribute_similarity_matrix.index.name = None
attribute_similarity_matrix.columns.name = None
attribute_similarity_matrix.to_csv(f'{output_path}/attribute_similarity_matrix_cosine.txt.gz', sep='\t', compression='gzip')
attribute_similarity_matrix

### Standardized Matrix

In [ ]:
standardized_matrix = pd.crosstab(mouse_associations['gene'], mouse_associations['motif'], values=mouse_associations['total_score'], aggfunc=sum).replace(np.nan, 0)
standardized_matrix.to_csv(f'{output_path}/gene_attribute_matrix_standardized.txt.gz', sep='\t', compression='gzip')
standardized_matrix

### Knowledge Graph Serialization

In [ ]:
nodes = {}
edges = []

for gene in geneslist.index:
    gene = geneslist.loc[gene]
    nodes[int(gene['Gene ID'])] = {
        "type":"gene",
        "properties": {
            "id":int(gene['Gene ID']),
            "label":gene['Gene']
        }}

for pert in attributeslist.index:
    pert = attributeslist.loc[pert]
    nodes[pert['TF ID']] = {
        "type":"transcription factor",
        "properties": {
            "label":pert['TF'],
            "id":pert['TF ID']
        }}

for edge in edgelist.index:
    edge = edgelist.loc[edge]
    edges.append({
        "source": str(edge['TF ID']),
        "relation": "directly regulates activity of",
        "target": int(edge['Gene ID']),
        "properties":{
            "id": str(edge['TF ID'])+":"+str(edge['Gene ID']),
            "source_label":edge['TF'],
            "source_id": str(edge['TF ID']),
            "target_id": int(edge['Gene ID']),
            "target_label":edge['Gene'],
            "directed":True,
            "threshold":1,
            "score":edge['Score']
        }})

#### RDF

In [ ]:
with open(output_path+'/kg_serializations/jasparpwmmouse25.rdf', 'w') as f:
    print('@prefix gene: <https://ncbi.nlm.nih.gov/gene/> .', file=f)
    print('@prefix RO: <http://purl.obolibrary.org/RO_> .', file=f)
    print('@prefix JASPAR: <https://jaspar.elixir.no/matrix/> .', file=f)
    
    print('', file=f)
    for edge in edges:
        source_id = edge['properties']['source_id']
        print(f"{gene if type(source_id)==int else 'JASPAR'}:{source_id}",'RO:0002448', f"gene:{edge['properties']['target_label']}", end=' .\n', file=f)

#### JSON

In [ ]:
with open(output_path+'/kg_serializations/jasparpwmmouse25.json', 'w') as f:
    serial = json.dump(
        {
            "Version":"1", 
            "nodes": nodes,
            "edges": edges
        }, indent=4, fp=f)

#### TSV

In [ ]:
def namespace(node_id):
    return 'NCBI Entrez' if type(node_id)==int else 'JASPAR'

In [ ]:
nodeframe = pd.DataFrame(nodes).T
nodeframe['id'] = nodeframe['properties'].apply(lambda x: x['id'])
nodeframe['label'] = nodeframe['properties'].apply(lambda x: x['label'])
#namespace = {'gene':'NCBI Entrez', 'gene perturbation':'Replogle et al., Cell, 2022'}
nodeframe['namespace'] = nodeframe['id'].map(namespace)
nodeframe = nodeframe.get(['namespace', 'id', 'label']).reset_index(drop=True)
nodeframe.to_csv(output_path+'/kg_serializations/jasparpwmmouse25_tsv/nodes.tsv', sep='\t')
nodeframe

In [ ]:
edgeframe = pd.DataFrame(edges)
edgeframe['threshold'] = edgeframe['properties'].apply(lambda x: x['threshold'])
edgeframe['score'] = edgeframe['properties'].apply(lambda x: x['score'])
edgeframe = edgeframe.get(['source', 'relation', 'target', 'score', 'threshold'])
edgeframe.to_csv(output_path+'/kg_serializations/jasparpwmmouse25_tsv/edges.tsv', sep='\t')
edgeframe

## Prepare Visualizations

### Gene-Attribute Clustermap

In [ ]:
sns.clustermap(binarymatrix, cmap='seismic', center=0, figsize=(10,10), xticklabels=False, yticklabels=False)

### Gene-Gene Similarity Clustermap

In [ ]:
gene_similarity_matrix = pd.read_csv('mouse_downloads/gene_similarity_matrix_cosine.txt.gz', sep='\t', compression='gzip', index_col=0)
gene_similarity_matrix

In [ ]:
ax = sns.clustermap(gene_similarity_matrix, cmap='seismic', center=0)

### Attribute-Attribute Similarity Clustermap

In [ ]:
sns.clustermap(attribute_similarity_matrix, cmap='seismic', center=0, xticklabels=False, yticklabels=False)

### UMAP

In [ ]:
def load_gmt(file):
    gmt = OrderedDict()
    for line in file:
        term, blank, *geneset = line.strip().split('\t')
        gmt[term] = ' '.join(set(geneset))
    return gmt
libdict = load_gmt(open(f'{output_path}/gene_set_library_crisp.gmt', 'r'))
scatterdir = 'mouse_images/'

In [ ]:
def process_scatterplot(libdict, nneighbors=30, mindist=0.1, spread=1.0, maxdf=1.0, mindf=1):
    print("\tTF-IDF vectorizing gene set data...")
    vec = TfidfVectorizer(max_df=maxdf, min_df=mindf)
    X = vec.fit_transform(libdict.values())
    print(X.shape)
    adata = anndata.AnnData(X)
    adata.obs.index = libdict.keys()

    print("\tPerforming Leiden clustering...")
    ### the n_neighbors and min_dist parameters can be altered
    sc.pp.neighbors(adata, n_neighbors=nneighbors, use_rep='X')
    sc.tl.leiden(adata, resolution=1.0)
    sc.tl.umap(adata, min_dist=mindist, spread=spread, random_state=42)

    new_order = adata.obs.sort_values(by='leiden').index.tolist()
    adata = adata[new_order, :]
    adata.obs['leiden'] = 'Cluster ' + adata.obs['leiden'].astype('object')

    df = pd.DataFrame(adata.obsm['X_umap'])
    df.columns = ['x', 'y']

    df['cluster'] = adata.obs['leiden'].values
    df['term'] = adata.obs.index
    df['genes'] = [libdict[l] for l in df['term']]

    return df

In [ ]:
def get_scatter_colors(df):
    clusters = pd.unique(df['cluster']).tolist()
    colors = list(Category20[20])[::2] + list(Category20[20])[1::2]
    color_mapper = {clusters[i]: colors[i % 20] for i in range(len(clusters))}
    return color_mapper

def get_scatterplot(scatterdf):
    df = scatterdf.copy()
    color_mapper = get_scatter_colors(df)
    df['color'] = df['cluster'].apply(lambda x: color_mapper[x])

    hover_emb = HoverTool(name="df", tooltips="""
        <div style="margin: 10">
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Gene Set:</span>
                <span style="font-size: 12px">@gene_set</span>
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Coordinates:</span>
                <span style="font-size: 12px">(@x,@y)</span>
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Cluster:</span>
                <span style="font-size: 12px">@cluster</span>
            </div>
        </div>
    """)
    tools_emb = [hover_emb, 'pan', 'wheel_zoom', 'reset', 'save']

    plot_emb = figure(
        width=1000, 
        height=700, 
        tools=tools_emb
    )

    source = ColumnDataSource(
        data=dict(
            x = df['x'],
            y = df['y'],
            gene_set = df['term'],
            cluster = df['cluster'],
            colors = df['color'],
            label = df['cluster']
        )
    )

    # hide axis labels and grid lines
    plot_emb.xaxis.major_tick_line_color = None
    plot_emb.xaxis.minor_tick_line_color = None
    plot_emb.yaxis.major_tick_line_color = None
    plot_emb.yaxis.minor_tick_line_color = None
    plot_emb.xaxis.major_label_text_font_size = '0pt'
    plot_emb.yaxis.major_label_text_font_size = '0pt' 

    plot_emb.output_backend = "svg"    
    
    plot_emb.title = 'Gene Sets in the JASPAR Predicted Mouse Transcription Factor Targets 2025 Library'
    plot_emb.xaxis.axis_label = "UMAP_1"
    plot_emb.yaxis.axis_label = "UMAP_2"
    plot_emb.xaxis.axis_label_text_font_style = 'normal'
    plot_emb.xaxis.axis_label_text_font_size = '18px'
    plot_emb.yaxis.axis_label_text_font_size = '18px'
    plot_emb.yaxis.axis_label_text_font_style = 'normal'
    plot_emb.title.align = 'center'
    plot_emb.title.text_font_size = '18px'
    
    s = plot_emb.scatter(
        'x', 
        'y', 
        size = 4, 
        source = source, 
        color = 'colors'
    )
    
    return plot_emb

In [ ]:
## defaults: nneighbors=30, mindist=0.1, spread=1.0, maxdf=1.0, mindf=1
scatter_df = process_scatterplot(libdict, nneighbors=7,mindist=0.1
     #,spread=0.5
     ,maxdf=0.5
     ,mindf=2
)

# Display Scatter Plot
plot = get_scatterplot(scatter_df)
output_notebook()
show(plot)

In [ ]:
output_file(filename=f"{scatterdir}/umap.html", title = 'Gene Sets in JASPAR Predicted Mouse Transcription Factor Targets 2025 Library')
save(plot)